# Notebook T4.5 — Pipeline de Moderación en Cascada
## Patrón: "Reglas → ML → LLM — el estándar de producción para clasificación a escala"

**Referencia en la guía metodológica:** PASO 5 → PATRÓN E  
**Nivel:** Intermedio

---

### El problema de coste en producción

Imagina que tienes 1 millón de comentarios al día que necesitas moderar.  
Si llamas al LLM para cada uno: 1.000.000 × $0.001 = **$1.000 al día** solo en API.

La solución: **pipeline en cascada**. Los casos obvios (la mayoría) se resuelven 
con métodos baratos y rápidos. El LLM solo procesa los casos difíciles.

```
100% comentarios → Reglas → 40% resueltos (gratis, 0ms)
                    60% restantes → ML → 45% resueltos (barato, 1ms)
                                   15% restantes → LLM → 15% resueltos ($$$, 2s)
```

**Ahorro: el LLM solo procesa el 15%** → coste reducido en ~85%.

---

### El patrón de diseño: `Optional[Resultado]`

Cada capa del pipeline usa el mismo contrato:
- Si **puede decidir con alta confianza** → devuelve un `ResultadoModeracion`
- Si **no está segura** → devuelve `None` y pasa a la siguiente capa

Esto se implementa con `Optional[ResultadoModeracion]` y la función retorna `None` 
cuando no puede decidir.

---

### Conexión con la guía metodológica

- **Paso 1:** Restricción de coste → obliga a usar cascada  
- **Paso 2:** Stack TF-IDF + Logistic Regression para la capa ML  
- **Paso 5:** PATRÓN E — Reglas → ML → LLM  
- **Paso 6:** Métricas del sistema: % por capa, ahorro de coste

## PASO 2 — Instalación y Configuración

In [ ]:
!pip install langchain langchain-google-genai scikit-learn pandas numpy python-dotenv -q

In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from dataclasses import dataclass
from typing import Optional
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0  # la moderación debe ser consistente
)
print("✅ Configuración lista")

## El Dataclass ResultadoModeracion

### ¿Qué es un dataclass?

Un `dataclass` es una clase Python simplificada para almacenar datos estructurados.  
En lugar de escribir `__init__` y `__repr__` manualmente, el decorador los genera automáticamente.

`ResultadoModeracion` es el objeto que todas las capas del pipeline devuelven.  
Tener un tipo de datos único para el resultado hace que el pipeline sea fácil de mantener:
- Cambiar los campos del resultado solo requiere cambiar el dataclass
- Cualquier capa que devuelve `None` no ha podido decidir — pasa a la siguiente

### `Optional[ResultadoModeracion]`

`Optional[X]` en Python type hints significa "puede ser X o puede ser None".  
Es la forma correcta de decir "esta función puede no devolver resultado".

In [ ]:
@dataclass
class ResultadoModeracion:
    """Resultado estándar que devuelve cualquiera de las capas del pipeline."""
    texto: str              # el texto analizado
    decision: str           # "rechazado" o "aprobado"
    capa: str               # "reglas", "ml", o "llm" — cuál capa tomó la decisión
    confianza: float        # entre 0.0 y 1.0 (cuán segura está la capa)
    razon: Optional[str] = None  # explicación de la decisión (opcional)

print("✅ Dataclass ResultadoModeracion definido")
print("\nEjemplo de uso:")
ejemplo = ResultadoModeracion(
    texto="Este es un comentario",
    decision="aprobado",
    capa="reglas",
    confianza=1.0,
    razon="Sin palabras prohibidas"
)
print(ejemplo)

## PASO 5 — Las 3 Capas del Pipeline

### Capa 1: Filtro por Reglas (gratuito, instantáneo)

La capa más simple. Busca palabras prohibidas en el texto.  
Si encuentra alguna → rechazado con confianza 1.0.  
Si no encuentra → devuelve `None` (no puede decidir, pasa al ML).

**Ventajas**: gratis, instantáneo, 100% determinista  
**Limitaciones**: solo detecta lo que está explícitamente en la lista — no entiende contexto

### Capa 2: Clasificador ML — TF-IDF + Regresión Logística

**TF-IDF (Term Frequency-Inverse Document Frequency)**: convierte texto en vectores numéricos.  
Da más peso a las palabras que aparecen frecuentemente en un documento pero raramente en otros.  
Es el enfoque clásico antes de los embeddings/LLMs.

**Pipeline de sklearn**: encadena TF-IDF + LogisticRegression en un solo objeto.  
Al hacer `pipeline.fit(textos, labels)`, primero aprende el vocabulario (TF-IDF) 
y luego entrena el clasificador sobre esos vectores.

**Umbral de confianza**: solo decide si `max(proba) >= 0.85`.  
Si no, devuelve `None` → caso ambiguo → pasa al LLM.

### Capa 3: LLM — Análisis contextual profundo (caro, lento)

La última línea de defensa. El LLM entiende contexto, ironía, matices culturales.  
Puede distinguir "este producto es una basura" (crítica válida) de un ataque personal.  
**Siempre decide** — es el fallback final del pipeline.

In [ ]:
PALABRAS_PROHIBIDAS = [
    "idiota", "imbécil", "inútil", "estúpido", "basura",
    "maldito", "imbecil", "hdp", "cretino", "animal"
]

def capa_reglas(texto: str) -> Optional[ResultadoModeracion]:
    """
    CAPA 1: Detección por palabras prohibidas.
    Retorna resultado si encuentra palabras explícitas, None si necesita análisis mayor.
    """
    texto_lower = texto.lower()
    palabras_encontradas = [p for p in PALABRAS_PROHIBIDAS if p in texto_lower]

    if palabras_encontradas:
        return ResultadoModeracion(
            texto=texto, decision="rechazado", capa="reglas", confianza=1.0,
            razon=f"Palabras prohibidas detectadas: {palabras_encontradas}"
        )
    return None  # No puede decidir con certeza → pasa a la siguiente capa


# --- DATASET DE ENTRENAMIENTO PARA LA CAPA ML ---
comentarios_train = [
    ("Eres un completo idiota, no sirves para nada", 1),
    ("Ojalá te pase algo malo, te lo mereces", 1),
    ("Imbécil, aprende antes de hablar", 1),
    ("Qué estupidez tan grande, inútil", 1),
    ("Me tienes harto con tus comentarios basura", 1),
    ("Eres un fraude y todos lo saben", 1),
    ("Cállate la boca, nadie quiere escucharte", 1),
    ("Tus ideas son una porquería", 1),
    ("Buen punto, no lo había considerado así", 0),
    ("Estoy en desacuerdo pero respeto tu opinión", 0),
    ("¿Podrías explicar mejor ese argumento?", 0),
    ("Interesante perspectiva, aunque yo lo veo diferente", 0),
    ("Gracias por compartir esta información", 0),
    ("Me parece que hay puntos a revisar en tu análisis", 0),
    ("Muy buen aporte al debate", 0),
    ("Creo que hay formas más constructivas de verlo", 0),
    ("Este producto es una basura total", 1),
    ("No entiendo cómo alguien puede ser tan torpe", 1),
    ("Qué decepción, esperaba más de ti", 0),
    ("Nunca más compro en esta tienda basura", 1),
]
df_train = pd.DataFrame(comentarios_train, columns=["texto", "toxico"])

# --- CAPA ML: TF-IDF + Regresión Logística ---
UMBRAL_CONFIANZA_ML = 0.85  # el ML solo decide si está muy seguro

# sklearn Pipeline encadena transformación + clasificador
# TF-IDF: texto → vector numérico; ngram_range=(1,2): usa palabras y bigrams
pipeline_ml = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ("clf", LogisticRegression(random_state=42, max_iter=1000))
])
pipeline_ml.fit(df_train["texto"], df_train["toxico"])

def capa_ml(texto: str) -> Optional[ResultadoModeracion]:
    """
    CAPA 2: Clasificador ML. Solo decide si la confianza supera el umbral del 85%.
    Retorna None si está inseguro → caso ambiguo → pasa al LLM.
    """
    proba = pipeline_ml.predict_proba([texto])[0]
    pred = int(np.argmax(proba))
    confianza = float(proba[pred])

    if confianza >= UMBRAL_CONFIANZA_ML:
        return ResultadoModeracion(
            texto=texto,
            decision="rechazado" if pred == 1 else "aprobado",
            capa="ml", confianza=confianza,
            razon=f"Clasificador ML (TF-IDF + LogReg) con confianza {confianza:.0%}"
        )
    return None  # Confianza insuficiente → pasa al LLM

# --- CAPA LLM: análisis contextual profundo ---
prompt_moderacion = ChatPromptTemplate.from_template("""
Eres un sistema de moderación de contenido preciso y justo.
Analiza este comentario y determina si debe ser rechazado.

Comentario: "{texto}"

Criterios para RECHAZAR:
- Ataques personales directos a una persona
- Amenazas o deseos explícitos de daño
- Lenguaje discriminatorio (racismo, sexismo, etc.)
- Acoso o intimidación

NO rechazar por:
- Críticas constructivas a productos o servicios
- Opiniones negativas expresadas respetuosamente
- Desacuerdo sin insultos directos

Responde ÚNICAMENTE con este JSON:
{{"decision": "rechazado" o "aprobado", "confianza": 0.0 a 1.0, "razon": "explicación breve de 1 oración"}}
""")

chain_moderacion = prompt_moderacion | llm | StrOutputParser()

def capa_llm(texto: str) -> ResultadoModeracion:
    """
    CAPA 3: LLM para análisis contextual. Siempre devuelve una decisión (es el fallback final).
    """
    raw = chain_moderacion.invoke({"texto": texto})
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    data = json.loads(raw)
    return ResultadoModeracion(
        texto=texto, decision=data["decision"], capa="llm",
        confianza=float(data["confianza"]), razon=data["razon"]
    )

print("✅ Las 3 capas del pipeline configuradas:")
print(f"  Capa 1 — Reglas: {len(PALABRAS_PROHIBIDAS)} palabras prohibidas")
print(f"  Capa 2 — ML:     TF-IDF + LogReg (umbral {UMBRAL_CONFIANZA_ML:.0%})")
print(f"  Capa 3 — LLM:    Gemini (fallback final)")

## PASO 8 — El Pipeline Completo y las Métricas del Sistema

In [ ]:
def moderar_comentario(texto: str) -> ResultadoModeracion:
    """
    Pipeline completo de moderación en cascada.
    Cada capa intenta decidir; si no puede (devuelve None), pasa a la siguiente.
    La Capa 3 (LLM) siempre decide y nunca devuelve None.
    """
    resultado = capa_reglas(texto)
    if resultado:
        return resultado   # Capa 1 decidió → terminamos aquí (rápido y gratis)

    resultado = capa_ml(texto)
    if resultado:
        return resultado   # Capa 2 decidió → terminamos aquí (barato)

    return capa_llm(texto)  # Capa 3 → siempre decide (caro, pero solo para casos difíciles)


# Casos de prueba diseñados para ilustrar cada capa
comentarios_test = [
    "Eres el peor idiota que he conocido",                    # → CAPA 1 (palabra prohibida)
    "Buen trabajo, me gustó mucho el artículo",               # → CAPA 2 ML (claramente no tóxico)
    "Ojalá te pase algo terrible, lo mereces",                # → CAPA 2 ML (patrón aprendido)
    "No estoy de acuerdo, pero lo respeto",                   # → CAPA 2 ML (claramente ok)
    "Qué decepción, esperaba mucho más de un profesional",    # → CAPA 3 LLM (ambiguo)
    "Esta empresa es una estafa, nunca más compro aquí",      # → CAPA 3 LLM (ambiguo)
    "Tu análisis ignora factores importantes, reconsidera",   # → CAPA 3 LLM (crítica constructiva)
]

print("🛡️ EJECUTANDO PIPELINE DE MODERACIÓN EN CASCADA")
print("=" * 70)

resultados = []
for comentario in comentarios_test:
    r = moderar_comentario(comentario)
    resultados.append(r)
    emoji = "🔴 RECHAZADO" if r.decision == "rechazado" else "✅ APROBADO"
    print(f"\n{emoji} — Capa: [{r.capa.upper()}] — Confianza: {r.confianza:.0%}")
    print(f"  Texto:  {comentario[:65]}...")
    print(f"  Razón:  {r.razon}")

In [ ]:
from collections import Counter

capas_usadas = Counter([r.capa for r in resultados])
decisiones = Counter([r.decision for r in resultados])
total = len(resultados)

print("\n" + "=" * 55)
print("📊 MÉTRICAS DEL PIPELINE (guía metodológica PASO 6)")
print("=" * 55)
print(f"\nTotal comentarios procesados: {total}")

print("\n🏗️ Distribución por capa:")
for capa, count in capas_usadas.most_common():
    bar = "█" * count
    pct = count / total
    cost_note = {"reglas": "(gratis)", "ml": "(barato)", "llm": "(costoso)"}[capa]
    print(f"  {capa:<8} {bar:<10} {count}/{total} ({pct:.0%}) {cost_note}")

print("\n🎯 Distribución por decisión:")
for decision, count in decisiones.items():
    print(f"  {decision:<12} {count}/{total} ({count/total:.0%})")

llm_calls = capas_usadas.get("llm", 0)
ahorro = (total - llm_calls) / total
print(f"\n💰 Llamadas al LLM (caro): {llm_calls}/{total} ({llm_calls/total:.0%})")
print(f"   Ahorro estimado vs usar solo LLM: {ahorro:.0%} del costo")
print(f"\n   → En 1 millón de comentarios/día: el LLM solo procesa {llm_calls/total:.0%}")
print(f"     Coste: ${1_000_000 * llm_calls/total * 0.001:,.0f}/día vs ${1_000_000 * 0.001:,.0f}/día si usaras solo LLM")

## Conclusiones del Notebook T4.5

### Lo que has aprendido

1. **PATRÓN E** — La cascada es el estándar de producción para clasificación a escala:  
   `Reglas (gratis) → ML (barato) → LLM (caro, solo casos ambiguos)`

2. **El patrón `Optional[Resultado]`**: cada capa devuelve resultado o `None`.  
   Este diseño hace el pipeline extensible — puedes añadir capas intermedias sin cambiar las demás.

3. **TF-IDF + LogReg = baseline clásico para clasificación de texto**: antes de los LLMs,  
   este era el estándar. Sigue siendo válido como capa 2 en cascadas por su velocidad.

4. **El umbral de confianza** es el parámetro clave de la capa ML:  
   - Umbral alto (0.95) → el ML decide poco → más casos llegan al LLM (más caro, más preciso)  
   - Umbral bajo (0.70) → el ML decide más → menos casos llegan al LLM (más barato, posibles errores)

5. **Métricas del sistema** (PASO 6 de la guía): en producción importa tanto la accuracy  
   como el coste, la latencia y la distribución por capas.

---

### El patrón de coste en cascada

| Capa | Velocidad | Coste por millón | Usa cuando |
|------|-----------|-----------------|------------|
| Reglas | <1ms | $0 | Casos con keywords explícitas |
| ML (TF-IDF) | ~1ms | ~$0.01 | Patrones aprendidos con alta confianza |
| LLM | ~2s | ~$1.00 | Casos ambiguos que necesitan razonamiento |

---

### Patrón clave (para memorizar)

```python
resultado = capa_reglas(texto) or capa_ml(texto) or capa_llm(texto)
# Cada capa retorna None si no puede decidir → la siguiente toma el relevo
```